# GoFlow — Couche d'intégration multi-agent

Une classe réutilisable reçoit des messages orchestrateur, réutilise les prévisions
XGBoost existantes et retourne une recommandation JSON. Aucun modèle d'incident,
d'urgence, de routes ou de transport public n'est créé. Les priorités sont des
choix **expérimentaux configurables**, pas la politique finale du groupe.

Les sorties affichées sont les résultats historiques conservés, non régénérés lors
du nettoyage. La démonstration finale utilise désormais `LocalLLMBackend`. Une
réexécution explicite de cette section tente un appel local si les fichiers sont
disponibles, sinon le fallback reste actif. Aucun résultat du dépôt n’est réécrit.


## 1. Charger l'agent et les contrats

`timestep` est obligatoire pour réaligner le forecasting historique. `timestamp`
peut rester null : aucune date de mesure PEMS08 n'est inventée.
Les contextes manquants restent inconnus. Les schémas et validations sont affichés
pour permettre aux autres membres du groupe d'adapter leurs messages.

Prérequis : installer ce dépôt dans l'environnement du noyau Jupyter, depuis sa
racine, avec `python -m pip install --no-index --no-deps --no-build-isolation -e .`.
Cette installation locale ne télécharge rien. Les imports fonctionnent ensuite
également depuis `notebooks/`, sans modification de `sys.path`.


In [1]:
# Importer la classe réutilisable sans exécuter les notebooks précédents.
from pathlib import Path
import json
import pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'notebooks/09_llm_integration.ipynb').exists())
from src.agents.traffic_prediction_agent import TrafficPredictionControlAgent
from src.contracts.traffic_prediction_contract import INPUT_SCHEMA, OUTPUT_SCHEMA
from src.decision.policy import DEFAULT_PRIORITY_POLICY as PRIORITY_POLICY
print('Politique expérimentale :', PRIORITY_POLICY)
print(json.dumps(INPUT_SCHEMA, ensure_ascii=False, indent=2))
print(json.dumps(OUTPUT_SCHEMA, ensure_ascii=False, indent=2))


Politique expérimentale : {'emergency': 100, 'incident': 80, 'severe_congestion': 60, 'public_transport': 40, 'normal': 20}
{
  "type": "object",
  "required": [
    "request_id",
    "sensor_id",
    "timestep",
    "traffic_data"
  ],
  "additionalProperties": false,
  "properties": {
    "request_id": {
      "type": "string",
      "minLength": 1,
      "maxLength": 200
    },
    "sensor_id": {
      "type": "integer",
      "minimum": 0,
      "maximum": 169
    },
    "timestep": {
      "type": "integer",
      "minimum": 15177,
      "maximum": 17853
    },
    "timestamp": {
      "type": [
        "string",
        "null"
      ]
    },
    "traffic_data": {
      "type": "object",
      "required": [
        "current_flow",
        "current_speed",
        "current_occupancy"
      ],
      "additionalProperties": false,
      "properties": {
        "current_flow": {
          "type": "number"
        },
        "current_speed": {
          "type": "number"
        },
    

## 2. Charger les données historiques

L’agent relit les prévisions existantes et construit le contexte causal et spatial.
Sans backend injecté, les sorties Qwen sauvegardées de 09 peuvent être rejouées
quand elles s’appliquent, puis validées. Les tests suivants ne lancent aucune
nouvelle génération et n’écrivent aucun journal ni résultat dans le dépôt.


In [2]:
# Initialiser les accès en lecture seule, sans configuration technique du backend.
agent = TrafficPredictionControlAgent(root)
assert agent._init_error is None, agent._init_error
print('Agent chargé : prévisions existantes, aucun fit et aucune nouvelle inférence.')


Agent chargé : prévisions existantes, aucun fit et aucune nouvelle inférence.


## 3. Tests avec MockOrchestrator

Les sept situations demandées sont couvertes, ainsi que conflits de priorité,
données incohérentes, timeout et garde-fous avant/après LLM. Les incidents, urgences,
priorités transport et backends artificiels sont explicitement marqués
`integration_test_fixture=true`. Les mesures de trafic restent issues des états réels.
Le test direct du garde-fou après LLM est distingué des échanges orchestrateur.


In [3]:
# Exécuter les tests de contrat ; MockOrchestrator reste dans le module de tests.
from tests.multi_agent_integration_cases import run_integration_cases
orchestrator, test_results = run_integration_cases(agent, root)
validation = pd.DataFrame(test_results)
display(validation)
assert validation.passed.all()
for message, response in zip(orchestrator.messages, orchestrator.responses):
    print(message.get('request_id'), '→', response['status'], response['recommended_action'],
          response['decision_source'])


,test,passed,input_valid_expected,input_schema_valid,response_contract_valid,status,decision_source,error_controlled,integration_test_fixture,llm_mode,critical_guard_before,critical_guard_after
0,trafic_normal,True,True,True,True,warning,rule_based_fallback,False,False,replay_saved_09_no_new_inference,False,False
1,forte_congestion_prevue,True,True,True,True,warning,rule_based_fallback,False,False,replay_saved_09_no_new_inference,False,False
2,incident_externe,True,True,True,True,warning,rule_based_fallback,False,True,not_called,True,True
3,urgence_externe,True,True,True,True,warning,rule_based_fallback,False,True,not_called,True,True
4,voisins_congestionnes,True,True,True,True,warning,rule_based_fallback,False,False,replay_saved_09_no_new_inference,False,False
5,contextes_absents,True,True,True,True,warning,rule_based_fallback,False,False,replay_saved_09_no_new_inference,False,False
6,message_invalide,True,False,False,True,error,rule_based_fallback,True,True,NaN,False,False
7,priorite_transport_public,True,True,True,True,warning,rule_based_fallback,False,True,not_called,False,False
8,urgence_prioritaire_sur_incident,True,True,True,True,warning,rule_based_fallback,False,True,not_called,True,True
9,trafic_modifie,True,False,True,True,error,rule_based_fallback,True,True,NaN,False,False


trafic_normal → warning keep_current_signal_plan rule_based_fallback
forte_congestion_prevue → warning coordinate_with_neighbor rule_based_fallback
incident_externe → warning manual_review rule_based_fallback
urgence_externe → warning manual_review rule_based_fallback
voisins_congestionnes → warning monitor rule_based_fallback
contextes_absents → warning keep_current_signal_plan rule_based_fallback
None → error manual_review rule_based_fallback
priorite_transport_public → warning manual_review rule_based_fallback
urgence_prioritaire_sur_incident → warning manual_review rule_based_fallback
trafic_modifie → error manual_review rule_based_fallback
bool_capteur → error manual_review rule_based_fallback
horizon_hors_export → error manual_review rule_based_fallback
boolean_externe_invalide → error manual_review rule_based_fallback
contexte_contradictoire → error manual_review rule_based_fallback
voisin_arbitraire → error manual_review rule_based_fallback
timestamp_sans_fuseau → error manual_

## 4. Examiner les garde-fous critiques

Une urgence externe ou un incident critique impose une revue manuelle, sans appel
LLM. Le contrôle après LLM réimpose la même décision. Un événement indiqué dans
une fixture reste une simulation d'interface, pas une observation PEMS08.


In [4]:
# Afficher un échange normal et un échange d'urgence simulée.
for request_id in ['trafic_normal', 'urgence_externe']:
    index = next(i for i, m in enumerate(orchestrator.messages) if m.get('request_id') == request_id)
    print(json.dumps({'message': orchestrator.messages[index], 'response': orchestrator.responses[index]},
                     ensure_ascii=False, indent=2, allow_nan=False))


{
  "message": {
    "request_id": "trafic_normal",
    "timestamp": null,
    "sensor_id": 33,
    "timestep": 15227,
    "traffic_data": {
      "current_flow": 0.0,
      "current_speed": 65.0,
      "current_occupancy": 0.0
    },
    "incident_context": {
      "incident_detected": null,
      "incident_type": null,
      "severity": null
    },
    "emergency_context": {
      "emergency_vehicle_present": null,
      "priority_level": null
    },
    "public_transport_context": {
      "priority_requested": null
    },
    "cooperative_context": {
      "neighboring_sensors": null
    },
    "integration_test_fixture": false
  },
  "response": {
    "agent": "traffic_prediction_control",
    "request_id": "trafic_normal",
    "sensor_id": 33,
    "forecast": {
      "horizon_minutes": 15,
      "predicted_occupancy": 0.0027397831
    },
    "traffic_state": "free_flow",
    "risk_level": "low",
    "recommended_action": "keep_current_signal_plan",
    "reason": "Occupation actuel

## 5. Résumé des scénarios historiques

Le tableau précédent et ce résumé sont calculés en mémoire. Les exports et la
[méthodologie historique](../results/multi_agent_integration/integration_methodology.md)
restent inchangés. Une erreur contrôlée attendue fait réussir un test négatif ;
le chemin LLM accepté simulé reste un test de contrat, pas une décision Qwen réelle.


In [5]:
# Résumer les échanges en mémoire ; ne réécrire aucun export historique.
summary = {
    'Agent intégré': 'oui' if validation.passed.all() else 'non',
    'Nombre de tests': len(validation), 'Tests réussis': int(validation.passed.sum()),
    'Tests échoués': int((~validation.passed).sum()),
    'Nombre de messages': len(orchestrator.messages),
    'Messages valides (schéma)': int(validation.input_schema_valid.fillna(False).sum()),
    'Messages exploitables (snapshot cohérent)': sum(r['status'] != 'error' for r in orchestrator.responses),
    'Réponses JSON valides': sum(agent.validate_response(r) for r in orchestrator.responses),
    'Fallbacks (réponses)': sum(r['decision_source'] == 'rule_based_fallback' for r in orchestrator.responses),
    'Erreurs contrôlées': sum(r['status'] == 'error' for r in orchestrator.responses),
    'Acceptations LLM simulées pour le contrat': sum(r['decision_source'] == 'llm' and r['metadata']['integration_test_fixture'] for r in orchestrator.responses)
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "Agent intégré": "oui",
  "Nombre de tests": 24,
  "Tests réussis": 24,
  "Tests échoués": 0,
  "Nombre de messages": 22,
  "Messages valides (schéma)": 16,
  "Messages exploitables (snapshot cohérent)": 14,
  "Réponses JSON valides": 22,
  "Fallbacks (réponses)": 21,
  "Erreurs contrôlées": 8,
  "Acceptations LLM simulées pour le contrat": 1
}


## 6. Architecture et périmètre de branchement

```text
Incident Detection Agent ──┐
Emergency Priority Agent ──┤
Public Transport Agent ────┤
Cooperative Traffic Agent ─┤
                          ↓
                     Orchestrator
                          ↓
          Traffic Prediction & Control Agent
                          ↓
              XGBoost forecast +15 existant
                          ↓
            LLM + garde-fous déterministes
                          ↓
               recommandation structurée
                          ↓
                     Orchestrator
```

L'interface est prête pour un branchement de test avec le contrat décrit. Cette démonstration
utilise le replay historique. Le mode live de l’agent exige un fournisseur
d’historique causal explicite ; aucun mode ne commande réellement les feux. Le mapping location=index NPZ reste
hypothétique pour 25 capteurs aux séries identiques. La priorité doit être validée
par l'équipe avant un usage opérationnel ; confidence n'est pas calibrée.


# DEMO FINALE — GOFLOW AGENT

**Un seul cas réel, un seul échange orchestrateur et un seul appel LLM si disponible.**
Les prévisions XGBoost sont déjà sauvegardées. Cette section ne réentraîne rien et
n'écrit aucun résultat. Les 24 tests et leurs sorties précédentes sont conservés.


## 1. Observation réelle

Choisir automatiquement la plus forte hausse d'occupation prévue parmi les lignes
test exportées. Le choix utilise la prévision et l'état courant, jamais la cible
réellement observée à t+3. Aucun incident ni véhicule d'urgence n'est inventé.


In [1]:
# Charger l'interface et sélectionner un vrai cas, indépendamment des cellules de tests.
from pathlib import Path
import json, copy
import pandas as pd
from IPython.display import display
demo_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'notebooks/09_llm_integration.ipynb').exists())
from src.agents.traffic_prediction_agent import TrafficPredictionControlAgent
from src.llm import LocalLLMBackend
from src.decision import guardrails
demo_agent = TrafficPredictionControlAgent(demo_root)
assert demo_agent._init_error is None, demo_agent._init_error
demo_rows = demo_agent.rules.PREDICTIONS
demo_changes = demo_rows.predicted_occupancy_t15 - demo_rows.current_occupancy
demo_index = demo_changes.idxmax()
assert demo_changes.loc[demo_index] > 0, 'Aucune hausse réelle disponible.'
demo_row = demo_rows.loc[demo_index]
demo_sensor, demo_timestep = int(demo_row.location), int(demo_row.timestep)
demo_base_state = demo_agent.rules.build_traffic_state(demo_sensor, demo_timestep)
display(pd.DataFrame([{key: demo_base_state[key] for key in
    ['sensor_id', 'timestep', 'current_flow', 'current_speed', 'current_occupancy',
     'predicted_occupancy_t15', 'occupancy_change']}]))


,sensor_id,timestep,current_flow,current_speed,current_occupancy,predicted_occupancy_t15,occupancy_change
0,134,16252,1.0,35.9,0.003,0.270631,0.267631


## 2. Prévision +15 min

**Prévision XGBoost à +15 minutes** — lecture d'une prévision existante, sans nouvelle
inférence XGBoost. Les valeurs sont dans l'échelle originale de l'occupation.


In [2]:
# Montrer la variation prévue sans modifier ou écrêter la prévision.
demo_forecast = demo_agent.predict(demo_sensor, demo_timestep)
assert demo_forecast == demo_base_state['predicted_occupancy_t15']
demo_delta = demo_forecast - demo_base_state['current_occupancy']
demo_direction = 'hausse prévue' if demo_delta > 0 else 'baisse prévue' if demo_delta < 0 else 'stable'
display(pd.DataFrame([{'current_occupancy': demo_base_state['current_occupancy'],
                      'predicted_occupancy_t15': demo_forecast, 'différence': demo_delta}]))
print('Interprétation :', demo_direction)


,current_occupancy,predicted_occupancy_t15,différence
0,0.003,0.270631,0.267631


Interprétation : hausse prévue


## 3. Contexte temporel et spatial

La tendance est la variation moyenne entre t−6 et t−1, par intervalle de 5 minutes :
positive signifie une hausse passée, négative une baisse passée. Les voisins sont
les connexions directes du graphe, sous l'hypothèse de mapping documentée pour
25 capteurs indiscernables. `timestamp=null` évite d'inventer une date PEMS08 ; le
`timestep` identifie le snapshot historique.


In [3]:
# Afficher les tendances causales et tous les voisins directs du cas sélectionné.
display(pd.DataFrame([{key: demo_base_state[key] for key in
                       ['recent_occupancy_trend', 'recent_speed_trend']}]))
for field, label in [('recent_occupancy_trend', 'occupation'), ('recent_speed_trend', 'vitesse')]:
    value = demo_base_state[field]
    direction = 'augmentait' if value > 0 else 'diminuait' if value < 0 else 'restait stable'
    print(f'Sur les six observations précédentes, {label} {direction} en moyenne.')
display(pd.DataFrame(demo_base_state['neighbors']))
demo_message = {
    'request_id': 'demo_final_001', 'timestamp': None, 'sensor_id': demo_sensor,
    'timestep': demo_timestep,
    'traffic_data': {key: demo_base_state[key] for key in
                     ['current_flow', 'current_speed', 'current_occupancy']},
    'incident_context': {'incident_detected': None, 'incident_type': None, 'severity': None},
    'emergency_context': {'emergency_vehicle_present': None, 'priority_level': None},
    'cooperative_context': {'neighboring_sensors': copy.deepcopy(demo_base_state['neighbors'])},
    'public_transport_context': {'priority_requested': None},
    'integration_test_fixture': False}
demo_state = demo_agent.build_state(demo_message)
print('Message envoyé à l’agent :')
print(json.dumps(demo_message, ensure_ascii=False, indent=2, allow_nan=False))


,recent_occupancy_trend,recent_speed_trend
0,-0.01334,-0.6


Sur les six observations précédentes, occupation diminuait en moyenne.
Sur les six observations précédentes, vitesse diminuait en moyenne.


,sensor_id,timestep,current_speed,current_occupancy,predicted_occupancy_t15
0,132,16252,32.9,0.1176,0.134004
1,136,16252,72.8,0.0022,0.034468


Message envoyé à l’orchestrateur simulé :
{
  "request_id": "demo_final_001",
  "timestamp": null,
  "sensor_id": 134,
  "timestep": 16252,
  "traffic_data": {
    "current_flow": 1.0,
    "current_speed": 35.9,
    "current_occupancy": 0.003
  },
  "incident_context": {
    "incident_detected": null,
    "incident_type": null,
    "severity": null
  },
  "emergency_context": {
    "emergency_vehicle_present": null,
    "priority_level": null
  },
  "cooperative_context": {
    "neighboring_sensors": [
      {
        "sensor_id": 132,
        "timestep": 16252,
        "current_speed": 32.9,
        "current_occupancy": 0.1176,
        "predicted_occupancy_t15": 0.13400355
      },
      {
        "sensor_id": 136,
        "timestep": 16252,
        "current_speed": 72.8,
        "current_occupancy": 0.0022,
        "predicted_occupancy_t15": 0.034467984
      }
    ]
  },
  "public_transport_context": {
    "priority_requested": null
  },
  "integration_test_fixture": false
}


## 4. Analyse LLM

Appeler le vrai Qwen2.5-0.5B-Instruct GGUF local via llama.cpp, avec le prompt et
le schéma de 09. Un serveur temporaire lié à localhost est arrêté après l'appel.
Son log reste temporaire : aucun fichier de résultats antérieur n'est écrit.
La réponse brute, même mauvaise, est affichée séparément de la décision finale.

`LocalLLMBackend` encapsule les fichiers locaux, le démarrage, les délais et l’arrêt.
Aucun téléchargement n’est automatique. Un modèle ou exécutable absent est
signalé dans les diagnostics et l’agent conserve son fallback déterministe.


In [4]:
# Le backend gère le serveur temporaire ; l'agent expose l'API de décision.
with LocalLLMBackend(demo_root) as demo_backend:
    demo_agent = TrafficPredictionControlAgent(demo_root, llm_backend=demo_backend)
    demo_response = demo_agent.handle_message(demo_message)
    demo_available = demo_backend.available

demo_call = demo_backend.last_call
print('Sortie LLM brute :')
print(demo_call['raw_text'] if demo_call['raw_text'] is not None else 'Aucune sortie : ' + str(demo_call['error']))
print('Temps d’inférence (s) :', demo_call['inference_time_s'])

demo_output = demo_call['parsed_output']
demo_json_valid = demo_agent.validate(demo_output)
print('JSON valide :', 'oui' if demo_json_valid else 'non')
if demo_call['error']:
    print('Erreur LLM conservée :', demo_call['error'])


C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sortie LLM brute :
{
  "traffic_state": "congested",
  "risk_level": "high",
  "recommended_action": "keep_current_signal_plan",
  "reason": "The current traffic is congested, and the predicted occupancy is high. It is recommended to keep the current signal plan in place.",
  "confidence": 0.99
}
Temps d’inférence (s) : 7.584002900053747
JSON valide : oui


## 5. Validation et garde-fous

Un JSON valide ne suffit pas : risque, action et justification doivent aussi
respecter les valeurs et le contrat enrichi. Les contrôles lexicaux d'incident et
d'urgence sont partiels ; une absence de terme suspect ne prouve pas une justesse
sémantique complète. Le grounding de l'agent impose ici sa justification de faits.


In [5]:
# Montrer chaque contrôle, sans remplacer la réponse produite par le modèle.
# Réutiliser les contrôles du backend et de l'agent, sans redéfinir la validation.
demo_checks = guardrails.validation_report(
    demo_output, demo_state, agent=demo_agent, rules=demo_agent.rules)

display(pd.DataFrame([{'Garde-fou': label, 'Résultat': 'OK' if passed else 'FAIL'}
                      for label, passed in demo_checks.items()]))
demo_fallback = demo_response['decision_source'] == 'rule_based_fallback'
if demo_fallback:
    print('LLM rejeté par les garde-fous' if demo_available else 'LLM indisponible — fallback déterministe')
    print('Raison :', demo_response['metadata'].get('fallback_reason'))
    if demo_response['metadata'].get('llm_error'):
        print('Erreur :', demo_response['metadata']['llm_error'])
    print('Décision rule_based utilisée :')
    print(json.dumps({key: demo_response[key] for key in
                      ['traffic_state', 'risk_level', 'recommended_action', 'reason', 'confidence']},
                     ensure_ascii=False, indent=2, allow_nan=False))
else:
    print('LLM accepté : aucun fallback nécessaire.')


,Garde-fou,Résultat
0,JSON valide,OK
1,traffic_state valide,OK
2,risk_level valide,OK
3,action autorisée,OK
4,raison suffisamment fondée,FAIL
5,aucune hallucination incident détectée,OK
6,aucune hallucination emergency détectée,OK
7,cohérence risque/action,FAIL
8,informations manquantes signalées,FAIL
9,contrat enrichi de l’agent respecté,FAIL


LLM rejeté par les garde-fous
Raison : llm_output_invalid_or_not_grounded
Décision rule_based utilisée :
{
  "traffic_state": "severe",
  "risk_level": "high",
  "recommended_action": "coordinate_with_neighbor",
  "reason": "Occupation actuelle=0.003000, prévue à +15 min=0.270631, variation=+0.267631; vitesse=35.900000; voisins à occupation élevée=1/2. Incident inconnu; urgence inconnue; priorité transport public inconnue; plan de feux indisponible.",
  "confidence": 0.5
}


## 6. Décision finale

### DÉCISION FINALE

La confidence n'est pas une probabilité calibrée. Cette recommandation reste une
sortie vers l'orchestrateur, sans commande de feux exécutée.


In [6]:
# Afficher seulement la décision finale, distincte de la réponse LLM brute.
demo_final = {key: demo_response[key] for key in
              ['traffic_state', 'risk_level', 'recommended_action', 'reason', 'confidence', 'decision_source']}
print(json.dumps(demo_final, ensure_ascii=False, indent=2, allow_nan=False))


{
  "traffic_state": "severe",
  "risk_level": "high",
  "recommended_action": "coordinate_with_neighbor",
  "reason": "Occupation actuelle=0.003000, prévue à +15 min=0.270631, variation=+0.267631; vitesse=35.900000; voisins à occupation élevée=1/2. Incident inconnu; urgence inconnue; priorité transport public inconnue; plan de feux indisponible.",
  "confidence": 0.5,
  "decision_source": "rule_based_fallback"
}


## 7. Réponse à l’orchestrateur

Voici la réponse complète réellement produite par `handle_message`, avec
prévision inchangée, corrélation request_id et trace de décision.


In [7]:
# La réponse complète est déjà validée par handle_message.
display(demo_response)


{
  "agent": "traffic_prediction_control",
  "request_id": "demo_final_001",
  "sensor_id": 134,
  "forecast": {
    "horizon_minutes": 15,
    "predicted_occupancy": 0.27063107
  },
  "traffic_state": "severe",
  "risk_level": "high",
  "recommended_action": "coordinate_with_neighbor",
  "reason": "Occupation actuelle=0.003000, prévue à +15 min=0.270631, variation=+0.267631; vitesse=35.900000; voisins à occupation élevée=1/2. Incident inconnu; urgence inconnue; priorité transport public inconnue; plan de feux indisponible.",
  "confidence": 0.5,
  "decision_source": "rule_based_fallback",
  "status": "warning",
  "metadata": {
    "timestep": 16252,
    "priority_policy_experimental": {
      "emergency": 100,
      "incident": 80,
      "severe_congestion": 60,
      "public_transport": 40,
      "normal": 20
    },
    "priority": "severe_congestion",
    "priority_rank": 60,
    "llm_mode": "injected_backend",
    "fallback_reason": "llm_output_invalid_or_not_grounded",
    "llm_er

### Résumé de la démonstration

Le modèle XGBoost prédit l'état futur du trafic. Le LLM propose une interprétation.
Les garde-fous vérifient la cohérence. Si nécessaire, le système utilise un fallback
déterministe. La décision finale est renvoyée à l'orchestrateur sous forme JSON structurée.


In [8]:
# Rapporter les résultats réellement observés de cette démonstration.
print('DEMO RESULT')
print('Sensor :', demo_sensor)
print('Timestep :', demo_timestep)
print('Forecast +15 :', demo_forecast)
print('LLM valid (JSON) :', 'oui' if demo_json_valid else 'non')
print('LLM accepté par les garde-fous :', 'oui' if not demo_fallback else 'non')
print('Fallback used :', 'oui' if demo_fallback else 'non')
print('Final action :', demo_response['recommended_action'])
print('Response status :', demo_response['status'])


DEMO RESULT
Sensor : 134
Timestep : 16252
Forecast +15 : 0.27063107
LLM valid (JSON) : oui
LLM accepté par les garde-fous : non
Fallback used : oui
Final action : coordinate_with_neighbor
Response status : warning
